#### Import das Biliotecas

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


#### Correção dos Exercícios

1) Crie um conjunto de dados pequeno (10 frases) de mensagens de SMS. Metade deve ser "Spam" (ex: "Ganhe prêmio agora", "Clique no link") e metade "Ham/Comum" (ex: "Oi, tudo bem?", "Vamos almoçar?").  

    **Tarefa**: Use o CountVectorizer para transformar as mensagens em vetores. Identifique quais são as 5 palavras que possuem a maior contagem no grupo de Spam e veja se elas fazem sentido intuitivo.  

    **Dica**: Tente classificar uma mensagem nova apenas comparando se ela tem mais palavras do "vocabulário spam" ou do "comum".


In [2]:
mensagens = [
    "Ganhe prêmio agora clique no link", 
    "Oferta imperdível ganhe dinheiro", 
    "Clique para receber seu prêmio",
    "Oi, tudo bem?", 
    "Vamos almoçar hoje?", 
    "Você viu a aula de PLN?" 
]

y = [1, 1, 1, 0, 0, 0] # 1: Spam, 0: Ham

vec = CountVectorizer()
X = vec.fit_transform(mensagens)

# Criando um DataFrame para visualizar as contagens por palavra
df = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
df['classe'] = y

# Palavras mais frequentes no Spam (classe 1)
spam_words = df[df['classe'] == 1].drop('classe', axis=1).sum().sort_values(ascending=False)
print("Top palavras no Spam:\n", spam_words.head(3))

Top palavras no Spam:
 clique    2
prêmio    2
ganhe     2
dtype: int64


2) O Desafio: Compare estas duas frases:
- "Eu gosto deste filme, não é ruim."
- "Eu não gosto deste filme, é ruim."

**Tarefa**: Gere os vetores BoW para ambas. Você notará que os vetores são idênticos (ambas têm as mesmas palavras).

**Solução**: Refaça o exercício usando o parâmetro ngram_range=(1, 2) no CountVectorizer. Observe como o vocabulário agora inclui "não gosto" e "não é", diferenciando as frases.


In [3]:
frases = [
    "Eu gosto deste filme não é ruim", 
    "Eu não gosto deste filme é ruim"
]

# Configurando para pegar palavras isoladas (1) e pares de palavras (2)
vec_bi = CountVectorizer(ngram_range=(1, 2))
X = vec_bi.fit_transform(frases)
vocabulario = vec_bi.get_feature_names_out()

# Exibindo apenas os Bigramas que fazem a diferença
print("Bigramas detectados que diferenciam as frases:")
for termo in vocabulario:
    if " " in termo: # Filtra para mostrar apenas os n-grams de tamanho 2
        print(f"- {termo}")

Bigramas detectados que diferenciam as frases:
- deste filme
- eu gosto
- eu não
- filme não
- filme ruim
- gosto deste
- não gosto
- não ruim


3) Pegue um texto longo (pode ser uma notícia ou um artigo)

**Tarefa**: Gere o Bag of Words e liste as 10 palavras mais frequentes. Provavelmente serão "o", "a", "de", "em" (stop words).

**Evolução**: Aplique uma lista de stop words em português e compare o tamanho do vetor antes e depois. Veja como o vocabulário "limpo" foca em substantivos e verbos que carregam o significado real do texto.


In [ ]:
texto = ["O aprendizado de máquina é uma subárea da inteligência artificial que foca em dados."]

#texto = ["aprendizado máquina subárea inteligência artificial foca dados"]

# Sem filtro
vec_ruidoso = CountVectorizer()
X_ruidoso = vec_ruidoso.fit_transform(texto)

# Com filtro (Exemplo de lista manual, já que o sklearn não tem nativo em PT-BR perfeito)
stop_pt = ['o', 'de', 'uma', 'da', 'que', 'em']
vec_limpo = CountVectorizer(stop_words=stop_pt)
X_limpo = vec_limpo.fit_transform(texto)

print(f"Colunas antes: {X_ruidoso.shape[1]} | Colunas depois: {X_limpo.shape[1]}")
print("Vocabulário útil:", vec_limpo.get_feature_names_out())

(1, 12)
Colunas antes: 12 | Colunas depois: 7
Vocabulário útil: ['aprendizado' 'artificial' 'dados' 'foca' 'inteligência' 'máquina'
 'subárea']


4) O Desafio: Implementar o Bag of Words sem usar o scikit-learn.

    **Tarefa**: Crie uma função que receba uma lista de frases.  
        a) Quebre as frases em palavras (tokenização).\
        b) Crie um dicionário único de palavras (vocabulário).  
        c) Retorne uma lista de listas (matriz) com as contagens. 

    Por que fazer? Isso ajuda a entender como lidar com pontuações e letras maiúsculas/minúsculas, algo que o CountVectorizer já faz automaticamente, mas que é vital conhecer.


In [9]:
def meu_bag_of_words(textos):

    # 1. Tokenização simples e criação do vocabulário
    vocabulario = sorted(list(set(" ".join(textos).lower().split())))
    word_to_idx = {palavra: i for i, palavra in enumerate(vocabulario)}
    
    # 2. Criação da matriz de contagem
    matriz = []
    for frase in textos:
        vetor = [0] * len(vocabulario)
        for palavra in frase.lower().split():
            if palavra in word_to_idx:
                vetor[word_to_idx[palavra]] += 1
        matriz.append(vetor)
        
    return np.array(matriz), vocabulario

# Teste
frases_teste = ["O rato roeu a roupa", "A roupa do rei"]
matriz, vocab = meu_bag_of_words(frases_teste)

print("Vocabulário:", vocab)
print("")
print(frases_teste)
print("")
print("Matriz BoW:\n", matriz)

Vocabulário: ['a', 'do', 'o', 'rato', 'rei', 'roeu', 'roupa']

['O rato roeu a roupa', 'A roupa do rei']

Matriz BoW:
 [[1 0 1 1 0 1 1]
 [1 1 0 0 1 0 1]]
